In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# 1. Definición de rutas
CAT = "databricks_proyecto_jhon"
BRONCE = f"{CAT}.lakehouse.bronce_ventas"
PLATA = f"{CAT}.lakehouse.plata_ventas"
CUAR = f"{CAT}.lakehouse.cuarentena_ventas"

df = spark.table(BRONCE)
total_entrada = df.count()

# 2. Deduplicación determinista (Nos quedamos con la versión más reciente si hay duplicados)
w = Window.partitionBy("id_venta").orderBy(F.col("_fecha_carga").desc())
limpio = (df
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1).drop("_rn"))

# 3. Normalización y Tipado Fuerte
limpio = (limpio
    .withColumn("estado", F.upper(F.trim(F.col("estado"))))
    .withColumn("canal", F.upper(F.trim(F.col("canal"))))
    .withColumn("ciudad", F.upper(F.trim(F.col("ciudad"))))
    .withColumn("categoria", F.upper(F.trim(F.col("categoria"))))
    .withColumn("fecha", F.to_date("fecha", "yyyy-MM-dd"))
    .withColumn("id_venta", F.col("id_venta").cast("int"))
    .withColumn("id_cliente", F.col("id_cliente").cast("int"))
    .withColumn("id_producto", F.col("id_producto").cast("int"))
    .withColumn("cantidad", F.col("cantidad").cast("int"))
    .withColumn("precio_unit_pen", F.col("precio_unit_pen").cast("double"))
    .withColumn("costo_unit_usd", F.col("costo_unit_usd").cast("double")))

# 4. Marcar registros inválidos (Sin eliminarlos todavía)
marcado = limpio.withColumn("_motivo",
    F.when(F.col("fecha").isNull(), F.lit("fecha_invalida"))
     .when(F.col("cantidad").isNull(), F.lit("cantidad_nula"))
     .when(F.col("cantidad") <= 0, F.lit("cantidad_no_positiva"))
     .when(F.col("precio_unit_pen").isNull(), F.lit("precio_nulo"))
     .when(F.col("precio_unit_pen") <= 0, F.lit("precio_no_positivo"))
     .when(F.col("costo_unit_usd").isNull(), F.lit("costo_nulo"))
     .when(~F.col("estado").isin("FACTURADA", "ANULADA", "PENDIENTE"), F.lit("estado_desconocido"))
     .otherwise(F.lit(None)))

validos = marcado.filter(F.col("_motivo").isNull()).drop("_motivo")
rechazados = marcado.filter(F.col("_motivo").isNotNull())

# 5. Cálculos de negocio para los datos válidos
validos = (validos
    .withColumn("monto_pen", F.round(F.col("cantidad") * F.col("precio_unit_pen"), 2))
    .withColumn("costo_usd", F.round(F.col("cantidad") * F.col("costo_unit_usd"), 2))
    .withColumn("anio", F.year("fecha")))

# 6. Guardado particionado (Preparando el terreno para optimización)
(validos.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("anio")
    .saveAsTable(PLATA))

# 7. Gestión de Cuarentena y Alertas
n_rec = rechazados.count()
if n_rec > 0:
    (rechazados.write.format("delta").mode("append")
        .option("mergeSchema", "true").saveAsTable(CUAR))

tasa = n_rec / total_entrada if total_entrada else 0
print(f"✅ [PLATA ventas] Entrada: {total_entrada:,} | Válidos: {validos.count():,} | Rechazados: {n_rec:,} ({tasa:.2%})")

if n_rec > 0:
    print("\n⚠️ Motivos de rechazo encontrados:")
    rechazados.groupBy("_motivo").count().show(truncate=False)

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

CAT = "databricks_proyecto_jhon"
PLATA_TC = f"{CAT}.lakehouse.plata_tipo_cambio_bcrp"

# 1. Leemos el histórico completo y desempaquetamos
tc_bronce = spark.table(f"{CAT}.lakehouse.bronce_tipo_cambio_bcrp")

tc = (tc_bronce
    .select(F.explode("periods").alias("p"))
    .select(
        F.col("p.name").alias("periodo"),
        F.col("p.values")[0].alias("tc_compra"),
        F.col("p.values")[1].alias("tc_venta")
    )
    # LA BARRERA: Eliminamos los días sin datos antes de hacer conversiones
    .filter((F.col("tc_compra") != "n.d.") & (F.col("tc_venta") != "n.d."))
)

# 2. Diccionario para traducir los meses que manda el BCRP
MESES = {"Ene": 1, "Feb": 2, "Mar": 3, "Abr": 4, "May": 5, "Jun": 6,
         "Jul": 7, "Ago": 8, "Set": 9, "Sep": 9, "Oct": 10, "Nov": 11, "Dic": 12}
mapa = F.create_map([F.lit(x) for kv in MESES.items() for x in kv])

# 3. Limpieza y formateo de la fecha sucia del BCRP ("15.Ene.25")
tc = (tc
    .withColumn("dia", F.regexp_extract("periodo", r"^(\d{1,2})", 1).cast("int"))
    .withColumn("mes_txt", F.regexp_extract("periodo", r"\.([A-Za-z]{3})\.", 1))
    .withColumn("anio_2d", F.regexp_extract("periodo", r"(\d{2})$", 1).cast("int"))
    .withColumn("mes", mapa[F.col("mes_txt")])
    .withColumn("anio", F.col("anio_2d") + F.lit(2000))
    .withColumn("fecha", F.make_date("anio", "mes", "dia"))
    .withColumn("tc_compra", F.col("tc_compra").cast("double"))
    .withColumn("tc_venta", F.col("tc_venta").cast("double"))
    .filter(F.col("fecha").isNotNull() & F.col("tc_venta").isNotNull())
    .select("fecha", "tc_compra", "tc_venta")
    .dropDuplicates(["fecha"]))

# 4. Magia de Ingeniería: Generar un calendario continuo sin huecos
limites = tc.agg(F.min("fecha").alias("ini"), F.max("fecha").alias("fin")).collect()[0]
calendario = spark.sql(f"SELECT explode(sequence(DATE '{limites['ini']}', DATE '{limites['fin']}', INTERVAL 1 DAY)) AS fecha")

# 5. Rellenar los fines de semana hacia adelante (Forward Fill)
w_ffill = Window.orderBy("fecha").rowsBetween(Window.unboundedPreceding, Window.currentRow)

tc_completo = (calendario
    .join(tc, "fecha", "left")
    .withColumn("tc_venta_ff", F.last("tc_venta", ignorenulls=True).over(w_ffill))
    .withColumn("tc_compra_ff", F.last("tc_compra", ignorenulls=True).over(w_ffill))
    # Bandera de honestidad: Marcamos si el BCRP no reportó dato ese día
    .withColumn("tc_es_imputado", F.col("tc_venta").isNull())
    .select(
        "fecha",
        F.col("tc_venta_ff").alias("tc_venta"),
        F.col("tc_compra_ff").alias("tc_compra"),
        "tc_es_imputado"
    ))

# 6. Guardamos en Plata
(tc_completo.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(PLATA_TC))

imputados = tc_completo.filter("tc_es_imputado").count()
print(f"✅ [PLATA tc] {tc_completo.count():,} días procesados | {imputados:,} imputados ({imputados / tc_completo.count():.1%})")